# Variant E (GDN) Sub-100M on T4Short-run setup for Kaggle T4 testing with checkpoint resume support.

In [ ]:
from pathlib import Pathimport jsonimport shleximport subprocessimport sysPROJECT_ROOT = Path.cwd()if PROJECT_ROOT.name.lower() == 'notebooks':    PROJECT_ROOT = PROJECT_ROOT.parentprint(f'Project root: {PROJECT_ROOT}')

In [ ]:
PRETOKENIZED_MANIFEST = PROJECT_ROOT / 'processed' / 'manifest.json'OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'variant_e_40m_t4'SIZE_PRESET = '40m'  # '40m' or '80m'MAX_PIECES = 15000EPOCHS = 6BATCH_SIZE = 1GRAD_ACCUM_STEPS = 16NUM_WORKERS = 2LEARNING_RATE = 2e-4SEED = 42LOG_EVERY_N_STEPS = 20RESUME_MODE = 'remaining'  # 'remaining' or 'additional'AUTO_RESUME = TrueALLOW_FALLBACK_GDN = TrueSEED_MIDI = ''  # optionaldef find_resume_checkpoint(checkpoint_dir: Path) -> str:    candidates = [        checkpoint_dir / 'latest_state.pt',        checkpoint_dir / 'latest.safetensors',        checkpoint_dir / 'best_state.pt',        checkpoint_dir / 'best.safetensors',    ]    for candidate in candidates:        if candidate.exists():            return str(candidate)    return ''OUTPUT_DIR.mkdir(parents=True, exist_ok=True)checkpoint_dir = OUTPUT_DIR / 'checkpoints' / f'variant_e_{SIZE_PRESET}'resume_ckpt = find_resume_checkpoint(checkpoint_dir) if AUTO_RESUME else ''print(f'Manifest: {PRETOKENIZED_MANIFEST}')print(f'Output: {OUTPUT_DIR}')print(f'Resume checkpoint: {resume_ckpt or "none"}')

In [ ]:
cmd = [    sys.executable, '-m', 'training.train_variant_e_sub100m',    '--size_preset', SIZE_PRESET,    '--pretokenized_manifest', str(PRETOKENIZED_MANIFEST),    '--output_dir', str(OUTPUT_DIR),    '--max_pieces', str(MAX_PIECES),    '--epochs', str(EPOCHS),    '--batch_size', str(BATCH_SIZE),    '--grad_accumulation_steps', str(GRAD_ACCUM_STEPS),    '--num_workers', str(NUM_WORKERS),    '--learning_rate', str(LEARNING_RATE),    '--seed', str(SEED),    '--log_every_n_steps', str(LOG_EVERY_N_STEPS),]if AUTO_RESUME:    cmd.extend(['--resume_from_checkpoint', 'auto', '--resume_mode', RESUME_MODE])if ALLOW_FALLBACK_GDN:    cmd.append('--allow_fallback_gdn')if str(SEED_MIDI).strip():    cmd.extend(['--seed_midi', str(SEED_MIDI)])print(shlex.join(cmd))

In [ ]:
RUN_TRAINING = Falseif RUN_TRAINING:    if not PRETOKENIZED_MANIFEST.exists():        raise FileNotFoundError(f'Manifest not found: {PRETOKENIZED_MANIFEST}')    subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)else:    print('Dry run only. Set RUN_TRAINING=True to launch.')result_path = OUTPUT_DIR / f'variant_e_{SIZE_PRESET}_result.json'if result_path.exists():    payload = json.loads(result_path.read_text(encoding='utf-8'))    print('Result path:', result_path)    print('Params:', payload.get('result', {}).get('params'))    val_loss = payload.get('result', {}).get('val_loss', [])    if val_loss:        print('Last val loss:', val_loss[-1])    print('Resume metadata:', payload.get('result', {}).get('resume', {}))    print('Backend status:', payload.get('backend_status', {}))else:    print('No result JSON yet.')